# Security & Secrets Management — Local Workflow

> **Mission:** Secure a Flask app with database credentials — zero secrets in git, images, or logs. Production-ready secrets handling.
>
> **This notebook:** 100% local secrets workflow (Docker Secrets, .env files, pre-commit hooks). We'll demonstrate bad practices, then fix them with runtime secret injection.
>
> **Supplement notebook:** Azure Key Vault for cloud-native secret management.

---

## Prerequisites

- **Docker Desktop** installed ([download](https://www.docker.com/products/docker-desktop))
- **Git** installed (for pre-commit hooks)
- **Trivy** installed for vulnerability scanning ([docs](https://aquasecurity.github.io/trivy/))

Verify installation:
```bash
docker --version
git --version
trivy --version
```

# Security & Secrets Management — Local Workflow

> **Mission:** Secure a Flask app with database credentials — zero secrets in git, images, or logs. Production-ready secrets handling.
>
> **This notebook:** 100% local secrets workflow (Docker Secrets, .env files, pre-commit hooks). We'll demonstrate bad practices, then fix them with runtime secret injection.
>
> **Supplement notebook:** Azure Key Vault for cloud-native secret management.

---

## Prerequisites

- **Docker Desktop** installed ([download](https://www.docker.com/products/docker-desktop))
- **Git** installed (for pre-commit hooks)
- **Trivy** installed for vulnerability scanning ([docs](https://aquasecurity.github.io/trivy/))

Verify installation:
```bash
docker --version
git --version
trivy --version
```

**Expected output:**
```
...
ENV API_KEY=sk_live_abcdef1234567890
ENV DB_PASSWORD=super_secret_password
...
```

❌ **Problem:** Secrets are permanently embedded in the image. Even if you delete the image and rebuild, the old image layers are still cached. If pushed to Docker Hub, the secrets are public forever.

✅ **Rule:** NEVER use `ENV` for secrets in Dockerfiles.

---

## Cell 2: Good Practice — Environment Variables (Local Dev Only)

**Goal:** Use environment variables passed at runtime instead of build-time secrets.

**What we're doing:**
- Create a Dockerfile **without** hardcoded secrets
- Pass secrets via `-e` flag at `docker run`
- Demonstrate that the image is now clean (no secrets in history)

**Why this works:** Secrets are provided at container startup, not baked into the image. The image can be pushed to a public registry safely.

In [ ]:
%%bash

# Create a secure Dockerfile (no secrets!)
mkdir -p /tmp/secure-app
cd /tmp/secure-app

cat > Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app

# No secrets in Dockerfile!
COPY . .
CMD ["python", "app.py"]
EOF

cat > app.py << 'EOF'
import os
api_key = os.getenv('API_KEY', 'NOT_SET')
db_password = os.getenv('DB_PASSWORD', 'NOT_SET')
print(f"API Key: {api_key}")
print(f"DB Password: {db_password}")
print("App running with runtime secrets...")
EOF

# Build the image
docker build -t secure-app:v1 .

echo ""
echo "====================="
echo "SECURITY AUDIT:"
echo "====================="
echo ""

# Verify no secrets in image
echo "Checking image history for secrets:"
docker history secure-app:v1 | grep -E "API_KEY|DB_PASSWORD" || echo "✅ No secrets found in image!"

echo ""
echo "Running container with runtime secrets:"
docker run --rm \
  -e API_KEY=sk_live_runtime_key \
  -e DB_PASSWORD=runtime_password \
  secure-app:v1

echo ""
echo "✅ Secrets passed at runtime, not in image!"

**Expected output:**
```
✅ No secrets found in image!
Running container with runtime secrets:
API Key: sk_live_runtime_key
DB Password: runtime_password
App running with runtime secrets...
```

✅ **Improvement:** Secrets are no longer in the image. However, `-e` secrets are visible in `docker inspect` and shell history. For production, use Docker Secrets or Kubernetes Secrets (mounted files).

---

## Cell 3: .env File + Docker Compose (Development)

**Goal:** Use `.env` files for local development secrets (never commit to git).

**What we're doing:**
- Create a `.env` file with secrets
- Create a `docker-compose.yml` that loads `.env`
- Demonstrate that secrets are loaded from the file, not hardcoded

**Why this works:** `.env` is gitignored. Developers can create their own `.env` locally without risking commits. Production uses different secret sources (Key Vault, Secrets Manager).

In [ ]:
%%bash

# Create project structure
mkdir -p /tmp/compose-app
cd /tmp/compose-app

# Create .env file (gitignored!)
cat > .env << 'EOF'
API_KEY=local_dev_key_123
DB_PASSWORD=local_dev_password_456
EOF

# Create .env.example (committed to git as template)
cat > .env.example << 'EOF'
API_KEY=your_api_key_here
DB_PASSWORD=your_db_password_here
EOF

# Create app code
cat > app.py << 'EOF'
import os
print(f"API Key: {os.getenv('API_KEY')}")
print(f"DB Password: {os.getenv('DB_PASSWORD')}")
print("✅ Secrets loaded from .env file!")
EOF

# Create Dockerfile (no secrets!)
cat > Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app
COPY app.py .
CMD ["python", "app.py"]
EOF

# Create docker-compose.yml
cat > docker-compose.yml << 'EOF'
services:
  app:
    build: .
    env_file:
      - .env  # Load secrets from .env (gitignored!)
EOF

# Build and run
docker-compose build
docker-compose up

# Clean up
docker-compose down

echo ""
echo "✅ Pattern: Commit .env.example, gitignore .env, developers create their own .env locally"

**Expected output:**
```
API Key: local_dev_key_123
DB Password: local_dev_password_456
✅ Secrets loaded from .env file!
```

✅ **Best practice for local dev:**
- **Commit** `.env.example` (template with placeholder values)
- **Gitignore** `.env` (actual secrets, never committed)
- **Developers** copy `.env.example` to `.env` and fill in their own secrets
- **Production** uses Docker Secrets, Kubernetes Secrets, or Key Vault (next cells)

---

## Cell 4: Docker Secrets (Swarm Mode)

**Goal:** Use Docker Secrets to mount secrets as files (more secure than environment variables).

**What we're doing:**
- Initialize Docker Swarm (required for Docker Secrets)
- Create a secret with `docker secret create`
- Deploy a service that reads the secret from `/run/secrets/`

**Why this is better:** Secrets are mounted as in-memory files, not environment variables. They never appear in `docker inspect`, process lists, or logs.

In [ ]:
%%bash

# Initialize Docker Swarm (required for Docker Secrets)
docker swarm init 2>/dev/null || echo "Swarm already initialized"

# Create a secret
echo "super_secret_db_password" | docker secret create db_password -

# Verify secret was created
docker secret ls

echo ""
echo "Creating app that reads Docker Secret..."

# Create app directory
mkdir -p /tmp/swarm-app
cd /tmp/swarm-app

# Create app that reads secret from file
cat > app.py << 'EOF'
import os

# Docker Secrets are mounted at /run/secrets/<secret_name>
secret_file = '/run/secrets/db_password'

try:
    with open(secret_file, 'r') as f:
        db_password = f.read().strip()
    print(f"✅ DB Password loaded from Docker Secret: {db_password}")
except FileNotFoundError:
    print(f"❌ Secret file not found: {secret_file}")
    print("(This is expected if not running in Swarm mode)")
EOF

# Create Dockerfile
cat > Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app
COPY app.py .
CMD ["python", "app.py"]
EOF

# Build image
docker build -t swarm-app:v1 .

# Deploy as a service with secret
docker service create \
  --name secure-service \
  --secret db_password \
  swarm-app:v1

# Wait for service to start
sleep 5

# Check service logs
docker service logs secure-service

# Clean up
docker service rm secure-service
docker secret rm db_password
docker swarm leave --force 2>/dev/null

echo ""
echo "✅ Docker Secrets are mounted as read-only files in /run/secrets/"
echo "✅ More secure than environment variables (not visible in docker inspect)"

**Expected output:**
```
✅ DB Password loaded from Docker Secret: super_secret_db_password
```

✅ **Production pattern:** Docker Secrets are encrypted at rest, transmitted over TLS, and mounted as in-memory tmpfs files. They never touch disk and are removed when the container stops.

**Note:** Docker Secrets require Swarm mode. For Kubernetes, use Kubernetes Secrets (next cell).

---

## Cell 5: Kubernetes Secrets (Base64 Encoding)

**Goal:** Use Kubernetes Secrets to inject secrets into pods (common in production clusters).

**What we're doing:**
- Create a Kubernetes Secret from a literal value
- Create a pod that loads the secret as an environment variable
- Demonstrate that secrets are base64-encoded (NOT encrypted)

**Why this matters:** K8s Secrets are base64-encoded by default. For production, enable encryption at rest with EncryptionConfiguration.

In [ ]:
%%bash

echo "Kubernetes Secret Example (YAML)"
echo "================================="
echo ""

# Example: Create a Kubernetes Secret
cat << 'EOF'
# Create secret from literal
$ kubectl create secret generic db-credentials \
    --from-literal=username=admin \
    --from-literal=password=super_secret_123

# Or create from YAML
apiVersion: v1
kind: Secret
metadata:
  name: db-credentials
type: Opaque
data:
  # Base64-encoded values (NOT encrypted!)
  username: YWRtaW4=          # echo -n 'admin' | base64
  password: c3VwZXJfc2VjcmV0XzEyMw==  # echo -n 'super_secret_123' | base64
EOF

echo ""
echo "---"
echo ""

# Example: Use secret in a pod
cat << 'EOF'
# Deployment that loads secrets
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flask-app
spec:
  template:
    spec:
      containers:
      - name: app
        image: flask-app:latest
        env:
        # Load secret as environment variable
        - name: DB_USERNAME
          valueFrom:
            secretKeyRef:
              name: db-credentials
              key: username
        - name: DB_PASSWORD
          valueFrom:
            secretKeyRef:
              name: db-credentials
              key: password
EOF

echo ""
echo "---"
echo ""
echo "⚠️  WARNING: Kubernetes Secrets are base64-encoded, NOT encrypted!"
echo "   Anyone with access to etcd can decode them:"
echo ""
echo "   $ kubectl get secret db-credentials -o jsonpath='{.data.password}' | base64 --decode"
echo "   super_secret_123"
echo ""
echo "✅ SOLUTION: Enable encryption at rest (EncryptionConfiguration) in your K8s cluster"

**Key takeaways:**
- **Base64 ≠ encryption**: Anyone with `kubectl` access can decode secrets
- **Encryption at rest**: Configure K8s with EncryptionConfiguration to encrypt secrets in etcd
- **RBAC**: Use Role-Based Access Control to restrict who can read secrets
- **Cloud-native alternatives**: Azure Key Vault, AWS Secrets Manager integrate with K8s for true secret management

✅ **Production pattern:** Use Kubernetes External Secrets Operator to sync from Key Vault/Secrets Manager into K8s Secrets.

---

## Cell 6: Scanning Images for Vulnerabilities (Trivy)

**Goal:** Use Trivy to scan Docker images for known vulnerabilities and exposed secrets.

**What we're doing:**
- Scan an image with Trivy (finds CVEs in dependencies)
- Scan for secrets with `trivy fs` (detects hardcoded credentials)
- Demonstrate automated security scanning in CI/CD

**Why this matters:** Vulnerabilities in base images (Python, Node, etc.) and accidentally committed secrets can be caught before deployment.

In [ ]:
%%bash

echo "Trivy Security Scanning Example"
echo "================================"
echo ""
echo "Install Trivy:"
echo "  brew install aquasecurity/trivy/trivy  # macOS"
echo "  choco install trivy                    # Windows"
echo "  apt-get install trivy                  # Linux"
echo ""
echo "---"
echo ""

# Check if Trivy is installed
if command -v trivy &> /dev/null; then
    echo "✅ Trivy is installed"
    
    # Scan our secure-app image for vulnerabilities
    echo ""
    echo "Scanning secure-app:v1 for vulnerabilities..."
    trivy image --severity HIGH,CRITICAL secure-app:v1 || echo "(No critical vulnerabilities found)"
    
    echo ""
    echo "---"
    echo ""
    echo "Scanning for secrets in codebase..."
    
    # Create a test file with a hardcoded secret
    mkdir -p /tmp/secret-scan
    echo "API_KEY=sk_live_abcdef1234567890" > /tmp/secret-scan/config.py
    
    # Scan for secrets
    trivy fs --scanners secret /tmp/secret-scan/ || echo "(Secrets detected!)"
    
    # Clean up
    rm -rf /tmp/secret-scan
else
    echo "⚠️  Trivy not installed. Install it to scan images:"
    echo ""
    echo "Example Trivy output:"
    echo ""
    cat << 'EOF'
$ trivy image python:3.11-slim

python:3.11-slim (debian 12.5)
Total: 45 (HIGH: 12, CRITICAL: 3)

┌────────────────┬────────────────┬──────────┬───────────────────┬───────────────┐
│    Library     │ Vulnerability  │ Severity │ Installed Version │ Fixed Version │
├────────────────┼────────────────┼──────────┼───────────────────┼───────────────┤
│ openssl        │ CVE-2024-1234  │ CRITICAL │ 1.1.1n            │ 1.1.1o        │
│ libcurl        │ CVE-2024-5678  │ HIGH     │ 7.74.0            │ 7.74.1        │
└────────────────┴────────────────┴──────────┴───────────────────┴───────────────┘

$ trivy fs --scanners secret .

config.py (secrets)
Total: 1 (HIGH: 1)

HIGH: AWS Access Key detected
────────────────────────────────
API_KEY=sk_live_abcdef1234567890
────────────────────────────────
EOF
fi

echo ""
echo "✅ CI/CD pattern: Run Trivy in GitHub Actions before pushing images"

**Expected output:**
```
HIGH: AWS Access Key detected
API_KEY=sk_live_abcdef1234567890
```

✅ **CI/CD integration:**
```yaml
# .github/workflows/security.yml
- name: Run Trivy scanner
  uses: aquasecurity/trivy-action@master
  with:
    image-ref: 'myapp:latest'
    format: 'sarif'
    output: 'trivy-results.sarif'
    severity: 'CRITICAL,HIGH'
```

**Why this matters:** Automated scanning catches vulnerabilities and leaked secrets before they reach production. Fail the build if HIGH/CRITICAL issues are found.

---

## Cell 7: Pre-Push Hook for Secret Detection

**Goal:** Install a Git pre-push hook that scans commits for secrets before pushing to remote.

**What we're doing:**
- Create a pre-push hook that searches for secret patterns
- Test it by attempting to commit a file with a hardcoded API key
- Block the push if secrets are detected

**Why this matters:** Prevents developers from accidentally pushing secrets to GitHub. Once secrets are in git history, they're permanently compromised (even if deleted later).

In [ ]:
%%bash

echo "Creating Git Pre-Push Hook for Secret Detection"
echo "================================================"
echo ""

# Create a test repo
mkdir -p /tmp/git-hook-demo
cd /tmp/git-hook-demo
git init

# Create pre-push hook
mkdir -p .git/hooks
cat > .git/hooks/pre-push << 'EOF'
#!/bin/bash

echo "🔍 Scanning for secrets before push..."

# Patterns to detect (adjust for your needs)
PATTERNS=(
    "password\s*=\s*['\"][^'\"]+['\"]"
    "api_key\s*=\s*['\"][^'\"]+['\"]"
    "secret\s*=\s*['\"][^'\"]+['\"]"
    "AWS_ACCESS_KEY_ID"
    "AZURE_CLIENT_SECRET"
    "sk_live_[a-zA-Z0-9]+"
)

# Scan staged files
for pattern in "${PATTERNS[@]}"; do
    if git diff --cached --diff-filter=ACMR | grep -iE "$pattern" > /dev/null; then
        echo "❌ ERROR: Potential secret detected!"
        echo "   Pattern matched: $pattern"
        echo ""
        echo "   Matched lines:"
        git diff --cached --diff-filter=ACMR | grep -iE "$pattern" --color=always
        echo ""
        echo "   PUSH BLOCKED. Remove secrets and try again."
        exit 1
    fi
done

echo "✅ No secrets detected. Push allowed."
exit 0
EOF

# Make hook executable
chmod +x .git/hooks/pre-push

# Test 1: Clean file (should pass)
echo "print('Hello, world!')" > app.py
git add app.py
git commit -m "Add clean file"
echo ""
echo "Test 1: Pushing clean file..."
git remote add origin https://github.com/example/demo.git 2>/dev/null || true
git push origin main 2>&1 | head -5 || echo "(This would succeed if remote existed)"

echo ""
echo "---"
echo ""

# Test 2: File with secret (should fail)
echo "API_KEY='sk_live_abcdef1234567890'" > config.py
git add config.py
git commit -m "Add config with API key"
echo ""
echo "Test 2: Pushing file with secret..."
git push origin main 2>&1 || echo "✅ Push blocked successfully!"

echo ""
echo "---"
echo ""
echo "✅ Install this hook in all repos:"
echo "   Copy .git/hooks/pre-push to your repo"
echo "   Or use a tool like pre-commit (https://pre-commit.com/)"

# Clean up
cd /tmp
rm -rf /tmp/git-hook-demo

**Expected output:**
```
Test 1: Pushing clean file...
✅ No secrets detected. Push allowed.

Test 2: Pushing file with secret...
❌ ERROR: Potential secret detected!
   Pattern matched: api_key\s*=\s*['"][^'"]+['"]
   PUSH BLOCKED. Remove secrets and try again.
```

✅ **Enterprise pattern:** Use tools like:
- **pre-commit** ([https://pre-commit.com/](https://pre-commit.com/)) — framework for managing git hooks
- **gitleaks** — specialized secret scanner with 1000+ patterns
- **GitHub Secret Scanning** — automatic scanning on push (free for public repos)

**Critical:** Pre-commit hooks are client-side. For enforcement, enable server-side scanning (GitHub Advanced Security, GitLab Secret Detection).

---

## Cell 8: Secrets Rotation Strategy

**Goal:** Demonstrate a zero-downtime secret rotation workflow.

**What we're doing:**
- Deploy two containers reading from a secret
- Update the secret in the secret store
- Restart containers with graceful rollout (one at a time)
- Verify no downtime during rotation

**Why this matters:** Secrets must be rotated regularly (every 90 days in most compliance frameworks). The rotation must not cause service outages.

In [ ]:
%%bash

echo "Secret Rotation Strategy (Zero Downtime)"
echo "========================================="
echo ""

cat << 'EOF'
Scenario: Rotate database password every 90 days without downtime

Step 1: Deploy app reading current secret
──────────────────────────────────────────
- App connects to database with password "old_password_v1"
- Multiple replicas running for high availability

Step 2: Generate new password
──────────────────────────────
- Generate new password: "new_password_v2"
- Update database to accept BOTH old and new passwords temporarily

Step 3: Update secret in secret store
──────────────────────────────────────
# Azure Key Vault
az keyvault secret set --vault-name myVault --name db-password --value "new_password_v2"

# Kubernetes
kubectl create secret generic db-credentials --from-literal=password="new_password_v2" --dry-run=client -o yaml | kubectl apply -f -

# Docker Swarm
echo "new_password_v2" | docker secret create db_password_v2 -

Step 4: Rolling update of containers
─────────────────────────────────────
# Kubernetes (automatic rolling update)
kubectl rollout restart deployment flask-app

# Docker Swarm (update service to use new secret)
docker service update --secret-rm db_password --secret-add db_password_v2 flask-app

Step 5: Verify all containers use new secret
─────────────────────────────────────────────
# Check logs to confirm new password is loaded
kubectl logs deployment/flask-app | grep "Connected to database"

Step 6: Remove old password from database
──────────────────────────────────────────
# Once all containers are using new password, revoke old one
# Database now accepts only "new_password_v2"

EOF

echo ""
echo "✅ Key principles:"
echo "   1. Dual-password window: Database accepts both old and new during transition"
echo "   2. Rolling updates: Replace containers one at a time (no downtime)"
echo "   3. Verification: Check logs before revoking old password"
echo "   4. Automation: Schedule rotation every 90 days with cron/workflow"
echo ""
echo "Example Kubernetes rolling update config:"
echo ""
cat << 'EOF'
spec:
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxUnavailable: 0  # Keep all pods running
      maxSurge: 1        # Start new pod before stopping old one
EOF

**Why zero downtime is possible:**
- **Rolling updates**: New containers start before old ones stop
- **Dual-password window**: Database accepts both passwords during transition
- **Health checks**: Kubernetes/Swarm waits for new container to be ready before stopping old one

✅ **Best practice:** Automate rotation with:
- **Azure Key Vault**: Automatic rotation policies
- **AWS Secrets Manager**: Lambda-based rotation functions
- **HashiCorp Vault**: Dynamic secrets with TTL (auto-rotation)

**Compliance:** SOC 2, PCI-DSS, HIPAA all require regular secret rotation (typically 90 days).

---

## Cell 9: RBAC — Least Privilege Principle

**Goal:** Demonstrate role-based access control for secrets (who can read what).

**What we're doing:**
- Define RBAC policies for Kubernetes Secrets
- Show how to restrict secret access to specific namespaces/pods
- Demonstrate the principle of least privilege

**Why this matters:** Not all containers should have access to all secrets. A frontend service doesn't need database credentials. RBAC limits blast radius if one service is compromised.

In [ ]:
%%bash

cat << 'EOF'
RBAC: Role-Based Access Control for Secrets
============================================

Principle: Grant each service ONLY the secrets it needs.

Example Architecture:
─────────────────────
- Frontend service: Needs API key for analytics (NOT database password)
- Backend service: Needs database password (NOT analytics API key)
- Admin pod: Needs ALL secrets for debugging

Kubernetes RBAC Example:
────────────────────────

1. Create namespace-specific secrets
────────────────────────────────────
# Frontend namespace
kubectl create namespace frontend
kubectl create secret generic analytics-key --namespace=frontend \
  --from-literal=api_key=GA_TRACKING_123

# Backend namespace
kubectl create namespace backend
kubectl create secret generic db-credentials --namespace=backend \
  --from-literal=password=db_password_456

2. Create ServiceAccount for each service
──────────────────────────────────────────
apiVersion: v1
kind: ServiceAccount
metadata:
  name: frontend-sa
  namespace: frontend
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: backend-sa
  namespace: backend

3. Create Role (what can be accessed)
──────────────────────────────────────
# Backend service can ONLY read db-credentials secret
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: secret-reader
  namespace: backend
rules:
- apiGroups: [""]
  resources: ["secrets"]
  resourceNames: ["db-credentials"]  # ONLY this secret!
  verbs: ["get"]

4. Create RoleBinding (who has the role)
─────────────────────────────────────────
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: backend-secret-binding
  namespace: backend
subjects:
- kind: ServiceAccount
  name: backend-sa
  namespace: backend
roleRef:
  kind: Role
  name: secret-reader
  apiGroup: rbac.authorization.k8s.io

5. Deploy pod with ServiceAccount
──────────────────────────────────
apiVersion: apps/v1
kind: Deployment
metadata:
  name: backend-app
  namespace: backend
spec:
  template:
    spec:
      serviceAccountName: backend-sa  # Uses this ServiceAccount
      containers:
      - name: app
        image: backend:latest
        env:
        - name: DB_PASSWORD
          valueFrom:
            secretKeyRef:
              name: db-credentials  # Can access ONLY this secret
              key: password

EOF

echo ""
echo "✅ Result:"
echo "   - Backend pod CAN read db-credentials secret"
echo "   - Backend pod CANNOT read analytics-key secret (different namespace)"
echo "   - Frontend pod CAN read analytics-key secret"
echo "   - Frontend pod CANNOT read db-credentials secret"
echo ""
echo "✅ Least privilege: Each service sees only its own secrets"
echo "   If frontend is compromised, database password is still safe"

**Key RBAC Concepts:**

| Concept | Purpose | Example |
|---------|---------|---------|
| **ServiceAccount** | Identity for a pod | `frontend-sa`, `backend-sa` |
| **Role** | Permissions within a namespace | "Can read secret X" |
| **RoleBinding** | Assigns role to ServiceAccount | "frontend-sa has role secret-reader" |
| **ClusterRole** | Permissions across all namespaces | (Use sparingly!) |

✅ **Cloud equivalents:**
- **Azure**: Managed Identity + Key Vault Access Policies
- **AWS**: IAM Roles + Secrets Manager Resource Policies
- **GCP**: Workload Identity + Secret Manager IAM

**Best practice:** Create a ServiceAccount per microservice, grant only required secrets via RBAC.

---

## Cell 10: Audit Logs — Who Accessed What Secret When

**Goal:** Enable audit logging to track secret access (compliance requirement).

**What we're doing:**
- Show how to enable audit logs in Kubernetes
- Demonstrate Azure Key Vault access logs
- Query logs to answer: "Who accessed the database password in the last 30 days?"

**Why this matters:** SOC 2, ISO 27001, HIPAA require audit trails. If a secret is compromised, you need to know who had access.

In [ ]:
%%bash

cat << 'EOF'
Audit Logging for Secrets Access
=================================

1. Kubernetes Audit Logs
─────────────────────────
Enable audit logging in K8s API server:

# /etc/kubernetes/audit-policy.yaml
apiVersion: audit.k8s.io/v1
kind: Policy
rules:
- level: RequestResponse
  resources:
  - group: ""
    resources: ["secrets"]
  verbs: ["get", "list"]

# API server flag
--audit-log-path=/var/log/kubernetes/audit.log
--audit-policy-file=/etc/kubernetes/audit-policy.yaml

Example audit log entry:
{
  "kind": "Event",
  "apiVersion": "audit.k8s.io/v1",
  "level": "RequestResponse",
  "user": {
    "username": "system:serviceaccount:backend:backend-sa"
  },
  "verb": "get",
  "objectRef": {
    "resource": "secrets",
    "namespace": "backend",
    "name": "db-credentials"
  },
  "responseStatus": {
    "code": 200
  },
  "requestReceivedTimestamp": "2024-04-26T10:30:00Z"
}

Query: "Who accessed db-credentials in the last 30 days?"
$ grep "db-credentials" /var/log/kubernetes/audit.log | jq '.user.username'

2. Azure Key Vault Audit Logs
──────────────────────────────
Enable diagnostic logs in Key Vault:

# Azure CLI
az monitor diagnostic-settings create \
  --resource /subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.KeyVault/vaults/{vault} \
  --name KeyVaultAudit \
  --logs '[{"category": "AuditEvent", "enabled": true}]' \
  --workspace {logAnalyticsWorkspaceId}

Query logs with KQL (Kusto Query Language):
AzureDiagnostics
| where ResourceType == "VAULTS"
| where OperationName == "SecretGet"
| where TimeGenerated > ago(30d)
| summarize Count=count() by CallerIPAddress, ClientId
| order by Count desc

Example output:
CallerIPAddress  ClientId                              Count
20.50.120.10     app-service-managed-identity          1234
10.0.1.5         developer@company.com                 45
40.80.100.20     ci-cd-pipeline-service-principal      890

3. AWS Secrets Manager CloudTrail
──────────────────────────────────
CloudTrail automatically logs all Secrets Manager API calls:

{
  "eventName": "GetSecretValue",
  "userIdentity": {
    "type": "AssumedRole",
    "principalId": "AIDAI...",
    "arn": "arn:aws:iam::123456789:role/backend-service-role"
  },
  "eventTime": "2024-04-26T10:30:00Z",
  "requestParameters": {
    "secretId": "prod/db/password"
  },
  "responseElements": null,
  "readOnly": true
}

Query: "Who accessed prod/db/password?"
$ aws cloudtrail lookup-events \
  --lookup-attributes AttributeKey=ResourceName,AttributeValue=prod/db/password \
  --start-time 2024-01-01

EOF

echo ""
echo "✅ Compliance checklist:"
echo "   ☑ Enable audit logging (K8s / Key Vault / Secrets Manager)"
echo "   ☑ Log secret access (who, what, when)"
echo "   ☑ Retain logs for 1+ year (compliance requirement)"
echo "   ☑ Alert on anomalies (unusual access patterns)"
echo "   ☑ Regular access reviews (quarterly audit)"
echo ""
echo "✅ Security incident response:"
echo "   If a secret is compromised, audit logs answer:"
echo "   - When was the secret first accessed?"
echo "   - Which services/users had access?"
echo "   - Was access authorized?"
echo "   - What data was exposed?"

**Audit log use cases:**

1. **Compliance audits**: Prove to auditors that only authorized services accessed production secrets
2. **Security incidents**: Determine blast radius of a compromised credential
3. **Access reviews**: Quarterly review of who has access to what secrets
4. **Anomaly detection**: Alert when a secret is accessed from an unusual IP or time

✅ **Enterprise pattern:**
- **Centralized logging**: Forward all audit logs to SIEM (Splunk, Azure Sentinel, AWS Security Hub)
- **Automated alerts**: Trigger alerts on high-risk events (secret accessed from unknown IP)
- **Retention**: Keep logs for 1-7 years (depends on compliance framework)

**Next steps:** Integrate audit logs with your security monitoring stack (Ch.5 Monitoring & Observability).